### 资产快照查询根据timestamp大小比较查询查不出来，因为只能查询最新的，所以要使用时间block查询

In [1]:
import requests
import time
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from IPython.display import display

# --- 1. 配置 Etherscan API 和网络 ---

# 你的 Etherscan API Key
ETHERSCAN_API_KEY = '85MDKMVQ9BIKM294IQY5N47QV1VN6SBXGP' 
BASE_URL = "https://api.etherscan.io/v2/api"

# 包含所有要检查的主网及其 Chain ID 的字典 (来自你的脚本)
networks_to_check = {
    "Base Mainnet": "8453",
    "Ethereum Mainnet": "1",
    "Abstract Mainnet": "2741",
    "ApeChain Mainnet": "33139",
    "Arbitrum Nova Mainnet": "42170",
    "Arbitrum One Mainnet": "42161",
    "Avalanche C-Chain": "43114",
    "Berachain Mainnet": "80094",
    "BitTorrent Chain Mainnet": "199",
    "Blast Mainnet": "81457",
    "BNB Smart Chain Mainnet": "56",
    "Celo Mainnet": "42220",
    "Fraxtal Mainnet": "252",
    "Gnosis": "100",
    "HyperEVM Mainnet": "999",
    "Katana Mainnet": "747474",
    "Linea Mainnet": "59144",
    "Mantle Mainnet": "5000",
    "Moonbeam Mainnet": "1284",
    "Moonriver Mainnet": "1285",
    "OP Mainnet": "10",
    "opBNB Mainnet": "204",
    "Polygon Mainnet": "137",
    "Scroll Mainnet": "534352",
    "Sei Mainnet": "1329",
    "Sonic Mainnet": "146",
    "Sophon Mainnet": "50104",
    "Swellchain Mainnet": "1923",
    "Taiko Mainnet": "167000",
    "Unichain Mainnet": "130",
    "World Mainnet": "480",
    "XDC Mainnet": "50",
    "zkSync Mainnet": "324"
}

# --- 2. 定义文件路径 ---

# 输入文件 (来自上一个 notebook)
INPUT_CSV_PATH = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\01_timerange_reserve\data\Target_sample.csv")

# 新的输出目录 (用于这个 notebook)
OUTPUT_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data")

# 最终的输出文件
OUTPUT_CSV_PATH = OUTPUT_DIR / "Target_sample_with_block_numbers.csv"


# --- 3. 定义 Etherscan 查询函数 ---

def get_tx_details_from_etherscan(tx_hash):
    """
    遍历所有定义的主网，查找给定 tx_hash 的网络和区块号。
    """
    for network_name, chain_id in networks_to_check.items():
        params = {
            'module': 'proxy',
            'action': 'eth_getTransactionReceipt',
            'txhash': tx_hash,
            'apikey': ETHERSCAN_API_KEY,
            'chainid': chain_id
        }
        
        try:
            response = requests.get(BASE_URL, params=params)
            response.raise_for_status()
            data = response.json()

            # 检查 API 是否返回了有效结果
            if data.get('result'):
                receipt = data['result']
                if receipt is None:
                    # 找到了但回执为空，继续下一个网络
                    continue
                
                block_number_hex = receipt.get('blockNumber')
                if block_number_hex:
                    block_number = int(block_number_hex, 16)
                    # 成功找到！
                    return network_name, block_number

        except requests.exceptions.RequestException as e:
            print(f"警告：请求 {network_name} API 时出错: {e}")
        except Exception as e:
            print(f"警告：处理 {network_name} 响应时出错: {e}")

        # 暂停 0.3 秒，避免触发速率限制
        time.sleep(0.3)
    
    # 如果循环结束都没有找到
    return None, None

# --- 4. 加载数据并处理 ---

print(f"正在从 {INPUT_CSV_PATH} 加载数据...")
try:
    df = pd.read_csv(INPUT_CSV_PATH)
except FileNotFoundError:
    print(f"错误：找不到输入文件 {INPUT_CSV_PATH}")
    print("请确保 'Target_sample.csv' 文件存在于该路径。")
    raise

print(f"成功加载 {len(df)} 条记录。")

# 用于存储结果的列表
found_networks = []
block_numbers = []

print("开始使用 Etherscan API 查询每笔交易的区块号...")

# 使用 tqdm 遍历 DataFrame 中的 txHash
for tx_hash in tqdm(df['txHash'], total=df.shape[0], desc="查询 Etherscan"):
    network, block = get_tx_details_from_etherscan(tx_hash)
    
    found_networks.append(network)
    block_numbers.append(block)

print("所有 Etherscan 查询已完成。")

# --- 5. 保存最终结果 ---

# 将新数据添加为 DataFrame 的列
df['found_network'] = found_networks
df['block_number'] = block_numbers

# 确保输出目录存在
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 保存到新的 CSV 文件
df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8')

print("\nDataFrame 更新完毕，包含网络和区块号：")
display(df.head())

print(f"\n✅ 数据已成功保存到: {OUTPUT_CSV_PATH}")

正在从 F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\01_timerange_reserve\data\Target_sample.csv 加载数据...
成功加载 61 条记录。
开始使用 Etherscan API 查询每笔交易的区块号...


查询 Etherscan: 100%|██████████| 61/61 [01:03<00:00,  1.04s/it]

所有 Etherscan 查询已完成。

DataFrame 更新完毕，包含网络和区块号：


,liquidation_timestamp,txHash,user_id,last_action_timestamp,last_action_type,liquidation_datetime,last_action_datetime,found_network,block_number
0,1761831755,0xa8e5f15f90e36332cda79e7087e45de1f487247ddb4e...,0xd8d6c693603729fdfed05a8243777ac6cb213508,1.761763e+09,liquidationCallHistory,2025-10-30 13:42:35,2025-10-29 18:41:07,Base Mainnet,37521204
1,1761831433,0x5080e4df6da8a0f4f0b0d4fd7e92c7b4b3a37721a9f0...,0x64d920358366a309c6d0363b361a18a7f81855ff,1.761584e+09,reserves,2025-10-30 13:37:13,2025-10-27 16:48:57,Base Mainnet,37521043
2,1761828723,0xa0a5a91430df9443b4565ed44c3d21afc1ae142d74dc...,0x33cf8f585e7063e31ec34f85721a65f4659f7172,1.761067e+09,reserves,2025-10-30 12:52:03,2025-10-21 17:19:03,Base Mainnet,37519688
3,1761828723,0xbb13e78bd55c4f8e30bb2026e6e9597af326c6a60ae3...,0x80a7dd43bf57aa72214578dc76857cf369243340,1.761754e+09,liquidationCallHistory,2025-10-30 12:52:03,2025-10-29 16:06:01,Base Mainnet,37519688
4,1761828573,0xc45a34f681ae8c1ec34edd9baebf705f941772f32775...,0xffc409d0074d411074ff282cb93ed5a2ffcafd76,1.760691e+09,liquidationCallHistory,2025-10-30 12:49:33,2025-10-17 08:53:55,Base Mainnet,37519613



✅ 数据已成功保存到: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\Target_sample_with_block_numbers.csv


# 将last_action_type按照非重复值分割为几个csv

In [3]:
import pandas as pd
from pathlib import Path
import os

# --- 1. 定义路径 ---

# 你的数据目录
DATA_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data")

# 上一个单元格生成的输入文件
INPUT_FILE = DATA_DIR / "Target_sample_with_block_numbers.csv"

# 你指定用于存放分割文件的新目录
SPLIT_OUTPUT_DIR = DATA_DIR / "last_operation_split"

# --- 2. 创建输出目录 ---

# 确保新目录存在
SPLIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"输出目录已准备好: {SPLIT_OUTPUT_DIR}")

# --- 3. 加载数据 ---

print(f"正在从 {INPUT_FILE} 加载数据...")
try:
    df = pd.read_csv(INPUT_FILE)
except FileNotFoundError:
    print(f"错误：找不到文件 {INPUT_FILE}")
    print("请确保上一个单元格已成功运行并生成了 CSV 文件。")
    # 如果文件不存在，停止执行
    raise

print(f"成功加载 {len(df)} 条记录。")

# --- 4. 查找唯一值并拆分 ---

# 检查拆分依据的列是否存在
if 'last_action_type' not in df.columns:
    print("错误：在 DataFrame 中找不到 'last_action_type' 列。")
    raise ValueError("Missing 'last_action_type' column")

# 获取 'last_action_type' 列中的所有非重复值
unique_action_types = df['last_action_type'].unique()

print(f"找到了 {len(unique_action_types)} 种唯一的 'last_action_type': {list(unique_action_types)}")

# 遍历每一种类型
for action_type in unique_action_types:
    
    # 筛选出该类型的所有行
    df_subset = df[df['last_action_type'] == action_type].copy()
    
    # 定义输出文件名 (例如: "reserves.csv")
    output_filename = f"{action_type}.csv"
    output_path = SPLIT_OUTPUT_DIR / output_filename
    
    # 保存子集
    df_subset.to_csv(output_path, index=False, encoding='utf-8')
    
    print(f"  -> 已保存 {len(df_subset)} 行到 {output_filename}")

print("\n✅ 所有文件已成功按 'last_action_type' 拆分。")

输出目录已准备好: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\last_operation_split
正在从 F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\Target_sample_with_block_numbers.csv 加载数据...
成功加载 61 条记录。
找到了 3 种唯一的 'last_action_type': ['liquidationCallHistory', 'reserves', 'userEmodeSetHistory']
  -> 已保存 18 行到 liquidationCallHistory.csv
  -> 已保存 42 行到 reserves.csv
  -> 已保存 1 行到 userEmodeSetHistory.csv

✅ 所有文件已成功按 'last_action_type' 拆分。


### 去调清算时候的资产快照

In [4]:
import requests
import json
import pandas as pd
from pathlib import Path
import time
from tqdm import tqdm
from IPython.display import display

# --- 1. 定义常量和路径 ---

# 你的 FastAPI 服务器 URL
FASTAPI_SERVER_URL = "http://localhost:8000/graphql"

# 输入文件 (来自上一个单元格的拆分结果)
INPUT_FILE = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\last_operation_split\reserves.csv")

# 你指定的新输出目录
OUTPUT_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\reserves_picture_liqudation")

# --- 2. 准备目录和加载数据 ---

# 确保输出目录存在
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"输出目录已准备好: {OUTPUT_DIR}")

print(f"正在从 {INPUT_FILE} 加载数据...")
try:
    df = pd.read_csv(INPUT_FILE)
except FileNotFoundError:
    print(f"错误：找不到文件 {INPUT_FILE}")
    print("请确保上一个单元格已成功运行并生成了 'reserves.csv'。")
    raise
    
# 检查必需的列
if 'user_id' not in df.columns or 'block_number' not in df.columns or 'txHash' not in df.columns:
    print("错误：CSV 文件中缺少 'user_id'、'block_number' 或 'txHash' 列。")
    raise ValueError("Missing required columns")

print(f"成功加载 {len(df)} 条 'reserves' 记录。")

# --- 3. 定义动态查询函数 ---

def build_snapshot_query(user_id, block_number):
    """
    根据用户ID和区块号，构建资产快照 GraphQL 查询。
    """
    # 确保 block_number 是整数
    block_num_int = int(block_number) 
    
    return f"""
    query GetUserSnapshot {{
      userReserves(
        where: {{user: "{user_id}"}}
        block: {{number: {block_num_int}}}
      ) {{
        currentATokenBalance
        currentTotalDebt
        usageAsCollateralEnabledOnUser
        reserve {{
          symbol
          decimals
          reserveLiquidationThreshold
          price {{
            priceInEth
          }}
        }}
      }}
    }}
    """

# --- 4. 遍历 DataFrame, 查询并保存 JSON 文件 ---

print(f"开始为 {len(df)} 条记录抓取资产快照...")

# 用于跟踪成功和失败的计数器
success_count = 0
fail_count = 0

# 使用 tqdm 显示进度条
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="抓取清算快照"):
    
    # 从行中获取关键信息
    user_id = row['user_id']
    block_number = row['block_number']
    tx_hash = row['txHash']
    
    # 检查 block_number 是否有效
    if pd.isna(block_number):
        print(f"警告：跳过 txHash {tx_hash}，因为 'block_number' 为空。")
        fail_count += 1
        continue
    
    # 1. 构建查询和 payload
    query_string = build_snapshot_query(user_id, block_number)
    payload = {"query": query_string}
    
    # 2. 发送请求
    try:
        # 注意：区块快照查询可能比常规查询慢，设置更长的超时时间
        response = requests.post(FASTAPI_SERVER_URL, json=payload, timeout=30)
        response.raise_for_status() # 检查 HTTP 错误
        data = response.json()
        
        # 3. 解析响应并构建最终的 JSON 对象
        if "errors" in data and data["errors"]:
            print(f"警告：GraphQL 查询失败 (txHash: {tx_hash})。错误: {data['errors'][0]['message']}")
            fail_count += 1
        
        elif "data" in data and "userReserves" in data["data"]:
            # 抓取快照数据
            snapshot_data = data["data"]["userReserves"]
            
            # 将整行 CSV 元数据转换为字典
            output_data = row.to_dict()
            
            # 将快照数据拼接到字典中
            output_data['liquidation_snapshot'] = snapshot_data
            
            # 4. 定义输出文件路径
            # 使用 txHash 作为唯一文件名
            output_filename = f"{tx_hash}.json"
            output_path = OUTPUT_DIR / output_filename
            
            # 5. 写入 JSON 文件
            with open(output_path, 'w', encoding='utf-8') as f:
                # indent=4 使 JSON 文件格式优美、易读
                json.dump(output_data, f, indent=4, ensure_ascii=False)
            
            success_count += 1
            
        else:
            print(f"警告：收到未知的响应格式 (txHash: {tx_hash})。")
            fail_count += 1
            
    except requests.exceptions.RequestException as e:
        print(f"警告：请求失败 (txHash: {tx_hash})。错误: {e}")
        fail_count += 1
    
    # 礼貌性暂停，避免请求过于频繁
    time.sleep(0.1)

print("\n--- 操作完成 ---")
print(f"✅ 成功抓取并保存了 {success_count} 个快照。")
print(f"❌ 失败或跳过了 {fail_count} 个记录。")
print(f"所有 JSON 文件已保存到: {OUTPUT_DIR}")

输出目录已准备好: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\reserves_picture_liqudation
正在从 F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\last_operation_split\reserves.csv 加载数据...
成功加载 42 条 'reserves' 记录。
开始为 42 条记录抓取资产快照...


抓取清算快照: 100%|██████████| 42/42 [01:44<00:00,  2.49s/it]


--- 操作完成 ---
✅ 成功抓取并保存了 42 个快照。
❌ 失败或跳过了 0 个记录。
所有 JSON 文件已保存到: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\reserves_picture_liqudation


单元格 1：重新抓取 清算前 (N-1) 的快照

这个单元格将读取 reserves.csv，但会查询 block_number - 1 并将结果保存到新文件夹 reserves_picture_PRE_liquidation。

In [5]:
import requests
import json
import pandas as pd
from pathlib import Path
import time
from tqdm import tqdm
from IPython.display import display

# --- 1. 定义常量和路径 ---
FASTAPI_SERVER_URL = "http://localhost:8000/graphql"
INPUT_FILE = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\last_operation_split\reserves.csv")

# *** 关键更改：使用新目录存储 "清算前" 快照 ***
OUTPUT_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\reserves_picture_PRE_liquidation")

# --- 2. 准备目录和加载数据 ---
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"输出目录已准备好: {OUTPUT_DIR} (用于存储'清算前'快照)")

print(f"正在从 {INPUT_FILE} 加载数据...")
try:
    df = pd.read_csv(INPUT_FILE)
except FileNotFoundError:
    print(f"错误：找不到文件 {INPUT_FILE}")
    raise
    
if 'user_id' not in df.columns or 'block_number' not in df.columns or 'txHash' not in df.columns:
    print("错误：CSV 文件中缺少 'user_id'、'block_number' 或 'txHash' 列。")
    raise ValueError("Missing required columns")

print(f"成功加载 {len(df)} 条 'reserves' 记录。")

# --- 3. 定义 *已更正* 的查询函数 ---

def build_PRE_snapshot_query(user_id, block_number):
    """
    (已更正)
    根据用户ID和区块号，构建 N-1 区块的资产快照 GraphQL 查询。
    """
    block_num_int = int(block_number)
    
    # ----------------- 关键修复 -----------------
    # 我们需要清算发生 *前* 的状态，即 N-1 区块
    previous_block = block_num_int - 1
    # ------------------------------------------
    
    return f"""
    query GetUserSnapshot {{
      userReserves(
        where: {{user: "{user_id}"}}
        block: {{number: {previous_block}}}  # <-- (已更正) 使用 N-10 区块号
      ) {{
        currentATokenBalance
        currentTotalDebt
        usageAsCollateralEnabledOnUser
        reserve {{
          symbol
          decimals
          reserveLiquidationThreshold
          price {{
            priceInEth
          }}
        }}
      }}
    }}
    """

# --- 4. 遍历 DataFrame, 查询并保存 JSON 文件 ---
print(f"开始为 {len(df)} 条记录抓取 *清算前* 资产快照...")

success_count = 0
fail_count = 0

for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="抓取 N-1 快照"):
    user_id = row['user_id']
    block_number = row['block_number']
    tx_hash = row['txHash']
    
    if pd.isna(block_number):
        print(f"警告：跳过 txHash {tx_hash}，因为 'block_number' 为空。")
        fail_count += 1
        continue
    
    # (已更正) 调用新的查询函数
    query_string = build_PRE_snapshot_query(user_id, block_number)
    payload = {"query": query_string}
    
    try:
        response = requests.post(FASTAPI_SERVER_URL, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        if "errors" in data and data["errors"]:
            print(f"警告：GraphQL 查询失败 (txHash: {tx_hash})。错误: {data['errors'][0]['message']}")
            fail_count += 1
        
        elif "data" in data and "userReserves" in data["data"]:
            snapshot_data = data["data"]["userReserves"]
            output_data = row.to_dict()
            output_data['pre_liquidation_snapshot'] = snapshot_data # 键名更改，更清晰
            
            output_filename = f"{tx_hash}.json"
            output_path = OUTPUT_DIR / output_filename
            
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(output_data, f, indent=4, ensure_ascii=False)
            
            success_count += 1
            
        else:
            print(f"警告：收到未知的响应格式 (txHash: {tx_hash})。")
            fail_count += 1
            
    except requests.exceptions.RequestException as e:
        print(f"警告：请求失败 (txHash: {tx_hash})。错误: {e}")
        fail_count += 1
    
    time.sleep(0.1)

print("\n--- 操作完成 ---")
print(f"✅ 成功抓取并保存了 {success_count} 个 '清算前' 快照。")
print(f"❌ 失败或跳过了 {fail_count} 个记录。")
print(f"所有 JSON 文件已保存到: {OUTPUT_DIR}")

输出目录已准备好: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\reserves_picture_PRE_liquidation (用于存储'清算前'快照)
正在从 F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\last_operation_split\reserves.csv 加载数据...
成功加载 42 条 'reserves' 记录。
开始为 42 条记录抓取 *清算前* 资产快照...


抓取 N-1 快照: 100%|██████████| 42/42 [01:43<00:00,  2.47s/it]


--- 操作完成 ---
✅ 成功抓取并保存了 42 个 '清算前' 快照。
❌ 失败或跳过了 0 个记录。
所有 JSON 文件已保存到: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\reserves_picture_PRE_liquidation


## 计算清算后每个样本的HF

好的，这个任务非常核心。我们将读取你上一步生成的每一个 JSON 文件，并根据 AAVE V3 协议的官方公式计算该用户在清算发生**当时**的健康因子（Health Factor, HF）。

### 🧮 健康因子 (Health Factor) 计算说明

AAVE 的健康因子 (HF) 是衡量一个账户清算风险的核心指标。如果 HF 低于 1，该账户就可以被清算。

其计算公式为：

$$
\text{Health Factor} = \frac{\sum (\text{CollateralInETH} \times \text{LiquidationThreshold})}{\sum (\text{TotalDebtInETH})}
$$

我将按以下步骤为你计算：

1.  **遍历资产 (Looping Assets):**
    我会遍历你 JSON 文件中 `liquidation_snapshot` 列表（即 `userReserves`）里的**每一项资产**。

2.  **计算分母 (Denominator): 总债务价值**
    * 对于**每一项资产**，我会读取其 `currentTotalDebt` (原始债务金额)。
    * 我会使用该资产的 `reserve.decimals` (小数位数) 将其转换为标准单位。
    * 我会读取 `reserve.price.priceInEth` (以 ETH 计价的价格)，并除以 $10^{18}$ (因为子图价格是以 18 位小数存储的) 来得到真实的 ETH 价格。
    * `TotalDebtInETH` = ( `currentTotalDebt` / $10^{\text{decimals}}$ ) $\times$ ( `priceInEth` / $10^{18}$ )
    * 我会将**所有资产**的 `TotalDebtInETH` 相加，得到分母。

3.  **计算分子 (Numerator): 加权总抵押价值**
    * 对于**每一项资产**，我**首先会检查 `usageAsCollateralEnabledOnUser` 是否为 `True`**。如果不是 `True`，该资产**不被视为抵押品**，其价值在分子中计为 0。
    * 如果为 `True`：
        * 我读取 `currentATokenBalance` (原始抵押金额) 并用 `reserve.decimals` 转换。
        * 我使用 `priceInEth` 计算其 `CollateralInETH` (方法同上)。
        * 我读取 `reserve.reserveLiquidationThreshold` (清算阈值，例如 "7800")。
        * 我将这个阈值转换为一个百分比因子 (例如：7800 / 10000 = 0.78)。
        * `WeightedCollateral` = `CollateralInETH` $\times$ `LiquidationThresholdFactor`
    * 我会将**所有被启用的抵押品**的 `WeightedCollateral` 相加，得到分子。

4.  **最终计算 (Final Calculation):**
    * `HF` = `加权总抵押价值` / `总债务价值`
    * 我会使用 Python 的 `Decimal` 库来处理这些计算，以确保金融计算的最高精度，**并按要求以标准小数格式（而非科学计数法）打印**。

---

### 单元格代码

请将以下代码复制到你的 `reserve_capture.ipynb` 笔记本的一个**新单元格**中运行。它会遍历 `reserves_picture_liqudation` 文件夹中的所有 JSON 文件并打印计算过程。

In [2]:
import json
import os
from pathlib import Path
from decimal import Decimal, getcontext

# --- 1. 设置高精度计算环境 ---
# 设置 Decimal 的精度为 50 位，足以应对金融计算
getcontext().prec = 50

# --- 2. 定义路径 ---
JSON_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\reserves_picture_liqudation")

# 1e18 常量，用于价格转换
PRICE_DECIMALS = Decimal('1e18') # 10**18
LT_DECIMALS = Decimal('10000')  # 清算阈值的小数位数

# --- 3. 检查目录是否存在 ---
if not JSON_DIR.exists():
    print(f"错误：找不到目录 {JSON_DIR}")
    print("请确保上一个单元格已成功运行并生成了 JSON 文件。")
    raise FileNotFoundError("JSON snapshot directory not found")

print(f"--- 开始计算 {JSON_DIR} 中所有样本的健康因子 ---")

# 获取目录中所有的 .json 文件
json_files = [f for f in os.listdir(JSON_DIR) if f.endswith('.json')]

if not json_files:
    print(f"警告：在 {JSON_DIR} 中未找到任何 .json 文件。")

# --- 4. 遍历所有 JSON 文件 ---
for filename in json_files:
    file_path = JSON_DIR / filename
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    # 提取元数据和快照数据
    tx_hash = data.get('txHash', 'N/A')
    user_id = data.get('user_id', 'N/A')
    snapshot_data = data.get('liquidation_snapshot', [])
    
    print(f"\n========================================================")
    print(f"正在计算: {filename}")
    print(f"TxHash: {tx_hash}")
    print(f"User: {user_id}")
    print(f"========================================================")

    total_collateral_weighted_eth = Decimal('0')
    total_debt_eth = Decimal('0')

    if not snapshot_data:
        print("  -> 警告：此样本中 'liquidation_snapshot' 为空。无法计算 HF。")
        continue

    # --- 5. 遍历该用户的所有资产快照 ---
    for asset in snapshot_data:
        try:
            # 提取公共数据
            symbol = asset['reserve']['symbol']
            decimals = int(asset['reserve']['decimals'])
            
            # 价格 (priceInEth) 是以 1e18 为基数的
            price_in_eth = Decimal(asset['reserve']['price']['priceInEth']) / PRICE_DECIMALS
            
            # --- 5a. 计算总债务 (分母) ---
            debt_amount_raw = Decimal(asset['currentTotalDebt'])
            if debt_amount_raw > 0:
                debt_amount = debt_amount_raw / (Decimal('10') ** decimals)
                debt_in_eth = debt_amount * price_in_eth
                total_debt_eth += debt_in_eth
                
                print(f"  资产: {symbol:<6} (债务)")
                print(f"    - 原始债务: {debt_amount_raw}")
                print(f"    - 价格 (ETH): {price_in_eth:.18f}")
                print(f"    - 债务价值 (ETH): {debt_in_eth:.18f}")

            # --- 5b. 计算加权抵押品 (分子) ---
            # 必须启用了 "useAsCollateral"
            if asset['usageAsCollateralEnabledOnUser']:
                collateral_amount_raw = Decimal(asset['currentATokenBalance'])
                
                if collateral_amount_raw > 0:
                    # 清算阈值 (例如 7800 -> 0.78)
                    lt_factor = Decimal(asset['reserve']['reserveLiquidationThreshold']) / LT_DECIMALS
                    
                    collateral_amount = collateral_amount_raw / (Decimal('10') ** decimals)
                    collateral_in_eth = collateral_amount * price_in_eth
                    weighted_collateral_eth = collateral_in_eth * lt_factor
                    
                    total_collateral_weighted_eth += weighted_collateral_eth
                    
                    print(f"  资产: {symbol:<6} (抵押品)")
                    print(f"    - 原始抵押: {collateral_amount_raw}")
                    print(f"    - 价格 (ETH): {price_in_eth:.18f}")
                    print(f"    - 抵押价值 (ETH): {collateral_in_eth:.18f}")
                    print(f"    - 清算阈值 (LT): {lt_factor:.4f}")
                    print(f"    - -> 加权价值 (ETH): {weighted_collateral_eth:.18f}")
            
        except Exception as e:
            print(f"  -> 错误：处理资产 {asset.get('reserve', {}).get('symbol', 'N/A')} 时出错: {e}")

    # --- 6. 打印最终 HF ---
    print(f"\n  --- 计算总计 ---")
    print(f"    加权总抵押 (分子): {total_collateral_weighted_eth:.18f} ETH")
    print(f"    总债务 (分母):   {total_debt_eth:.18f} ETH")
    
    if total_debt_eth == Decimal('0'):
        # 如果没有债务，HF 是无限大
        print(f"  >>> 最终健康因子 (HF): Infinity (无债务)")
    else:
        try:
            health_factor = total_collateral_weighted_eth / total_debt_eth
            # 按要求使用非科学计数法打印
            print(f"  >>> 最终健康因子 (HF): {health_factor:.18f}")
            if health_factor < Decimal('1.0'):
                print("  (状态：可清算)")
        except Exception as e:
            print(f"  >>> 最终健康因子 (HF): 计算出错 ({e})")

print("\n--- 所有文件计算完毕 ---")

--- 开始计算 F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\reserves_picture_liqudation 中所有样本的健康因子 ---

正在计算: 0x5080e4df6da8a0f4f0b0d4fd7e92c7b4b3a37721a9f07516c9e14cdbdacba885.json
TxHash: 0x5080e4df6da8a0f4f0b0d4fd7e92c7b4b3a37721a9f07516c9e14cdbdacba885
User: 0x64d920358366a309c6d0363b361a18a7f81855ff
  资产: cbETH  (抵押品)
    - 原始抵押: 862685721914231983
    - 价格 (ETH): 0.000000000000000000
    - 抵押价值 (ETH): 0.000000000000000000
    - 清算阈值 (LT): 0.7900
    - -> 加权价值 (ETH): 0.000000000000000000
  资产: WETH   (抵押品)
    - 原始抵押: 100590084925665881
    - 价格 (ETH): 0.000000378581000000
    - 抵押价值 (ETH): 0.000000038081494941
    - 清算阈值 (LT): 0.8300
    - -> 加权价值 (ETH): 0.000000031607640801
  资产: USDC   (债务)
    - 原始债务: 3455866215
    - 价格 (ETH): 0.000000000099987000
    - 债务价值 (ETH): 0.000000345541695239
  资产: cbBTC  (债务)
    - 原始债务: 1
    - 价格 (ETH): 0.000008155695500000
    - 债务价值 (ETH): 0.000000000000081557
  资产:

## 重新计算 HF（使用 清算前 快照）

In [6]:
import json
import os
from pathlib import Path
from decimal import Decimal, getcontext

# --- 1. 设置高精度计算环境 ---
getcontext().prec = 50

# --- 2. (已更正) 定义路径 ---
# *** 关键更改：读取包含 "清算前" 快照的新目录 ***
JSON_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\reserves_picture_PRE_liquidation")

PRICE_DECIMALS = Decimal('1e18') # 10**18
LT_DECIMALS = Decimal('10000')  # 清算阈值的小数位数

# --- 3. 检查目录是否存在 ---
if not JSON_DIR.exists():
    print(f"错误：找不到目录 {JSON_DIR}")
    print("请确保上一个单元格已成功运行并生成了 JSON 文件。")
    raise FileNotFoundError("JSON snapshot directory not found")

print(f"--- 开始计算 {JSON_DIR} 中所有样本的 *清算前* 健康因子 ---")

json_files = [f for f in os.listdir(JSON_DIR) if f.endswith('.json')]

if not json_files:
    print(f"警告：在 {JSON_DIR} 中未找到任何 .json 文件。")

# --- 4. 遍历所有 JSON 文件 ---
for filename in json_files:
    file_path = JSON_DIR / filename
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    tx_hash = data.get('txHash', 'N/A')
    user_id = data.get('user_id', 'N/A')
    
    # *** 关键更改：读取 'pre_liquidation_snapshot' 键 ***
    snapshot_data = data.get('pre_liquidation_snapshot', [])
    
    print(f"\n========================================================")
    print(f"正在计算: {filename}")
    print(f"TxHash: {tx_hash}")
    print(f"User: {user_id}")
    print(f"========================================================")

    total_collateral_weighted_eth = Decimal('0')
    total_debt_eth = Decimal('0')

    if not snapshot_data:
        print("  -> 警告：此样本中 'pre_liquidation_snapshot' 为空。无法计算 HF。")
        continue

    # --- 5. 遍历该用户的所有资产快照 ---
    for asset in snapshot_data:
        try:
            symbol = asset['reserve']['symbol']
            decimals = int(asset['reserve']['decimals'])
            price_in_eth = Decimal(asset['reserve']['price']['priceInEth']) / PRICE_DECIMALS
            
            # --- 5a. 计算总债务 (分母) ---
            debt_amount_raw = Decimal(asset['currentTotalDebt'])
            if debt_amount_raw > 0:
                debt_amount = debt_amount_raw / (Decimal('10') ** decimals)
                debt_in_eth = debt_amount * price_in_eth
                total_debt_eth += debt_in_eth
                
                print(f"  资产: {symbol:<6} (债务)")
                print(f"    - 原始债务: {debt_amount_raw}")
                print(f"    - 价格 (ETH): {price_in_eth:.18f}")
                print(f"    - 债务价值 (ETH): {debt_in_eth:.18f}")

            # --- 5b. 计算加权抵押品 (分子) ---
            if asset['usageAsCollateralEnabledOnUser']:
                collateral_amount_raw = Decimal(asset['currentATokenBalance'])
                
                if collateral_amount_raw > 0:
                    lt_factor = Decimal(asset['reserve']['reserveLiquidationThreshold']) / LT_DECIMALS
                    collateral_amount = collateral_amount_raw / (Decimal('10') ** decimals)
                    collateral_in_eth = collateral_amount * price_in_eth
                    weighted_collateral_eth = collateral_in_eth * lt_factor
                    
                    total_collateral_weighted_eth += weighted_collateral_eth
                    
                    print(f"  资产: {symbol:<6} (抵押品)")
                    print(f"    - 原始抵押: {collateral_amount_raw}")
                    print(f"    - 价格 (ETH): {price_in_eth:.18f}")
                    print(f"    - 抵押价值 (ETH): {collateral_in_eth:.18f}")
                    print(f"    - 清算阈值 (LT): {lt_factor:.4f}")
                    print(f"    - -> 加权价值 (ETH): {weighted_collateral_eth:.18f}")
            
        except Exception as e:
            print(f"  -> 错误：处理资产 {asset.get('reserve', {}).get('symbol', 'N/A')} 时出错: {e}")

    # --- 6. 打印最终 HF ---
    print(f"\n  --- 计算总计 (清算前) ---")
    print(f"    加权总抵押 (分子): {total_collateral_weighted_eth:.18f} ETH")
    print(f"    总债务 (分母):   {total_debt_eth:.18f} ETH")
    
    if total_debt_eth == Decimal('0'):
        print(f"  >>> 最终健康因子 (HF): Infinity (无债务)")
    else:
        try:
            health_factor = total_collateral_weighted_eth / total_debt_eth
            print(f"  >>> 最终健康因子 (HF): {health_factor:.18f}")
            if health_factor < Decimal('1.0'):
                print("  (状态：可清算 - 这是预期的结果！)")
            else:
                print("  (状态：不可清算 - 警告：HF仍大于1，请检查数据)")
        except Exception as e:
            print(f"  >>> 最终健康因子 (HF): 计算出错 ({e})")

print("\n--- 所有文件计算完毕 ---")

--- 开始计算 F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\reserves_picture_PRE_liquidation 中所有样本的 *清算前* 健康因子 ---

正在计算: 0x5080e4df6da8a0f4f0b0d4fd7e92c7b4b3a37721a9f07516c9e14cdbdacba885.json
TxHash: 0x5080e4df6da8a0f4f0b0d4fd7e92c7b4b3a37721a9f07516c9e14cdbdacba885
User: 0x64d920358366a309c6d0363b361a18a7f81855ff
  资产: cbETH  (抵押品)
    - 原始抵押: 862685721914231983
    - 价格 (ETH): 0.000000000000000000
    - 抵押价值 (ETH): 0.000000000000000000
    - 清算阈值 (LT): 0.7900
    - -> 加权价值 (ETH): 0.000000000000000000
  资产: WETH   (抵押品)
    - 原始抵押: 100590084925665881
    - 价格 (ETH): 0.000000378581000000
    - 抵押价值 (ETH): 0.000000038081494941
    - 清算阈值 (LT): 0.8300
    - -> 加权价值 (ETH): 0.000000031607640801
  资产: USDC   (债务)
    - 原始债务: 6817096739
    - 价格 (ETH): 0.000000000099987000
    - 债务价值 (ETH): 0.000000681621051642
  资产: cbBTC  (债务)
    - 原始债务: 1
    - 价格 (ETH): 0.000008155695500000
    - 债务价值 (ETH): 0.000000000000

## 我认为可能还有其他的清算的条件，导致不是健康因子的原因就会被清算

In [7]:
import json
import os
from pathlib import Path
from decimal import Decimal, getcontext

# --- 1. 设置高精度计算环境 ---
getcontext().prec = 50

# --- 2. 定义所有路径 ---

# 输入目录 (包含 N-1 快照)
INPUT_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\reserves_picture_PRE_liquidation")

# 输出目录 (正常样本)
REACH_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\Reach_and_unReach_sample\Reach")

# 输出目录 (异常样本)
UNREACH_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\Reach_and_unReach_sample\UnReach")

# AAVE 计算常量
PRICE_DECIMALS = Decimal('1e18') # 10**18
LT_DECIMALS = Decimal('10000')  # 清算阈值的小数位数

# --- 3. 确保输出目录存在 ---
REACH_DIR.mkdir(parents=True, exist_ok=True)
UNREACH_DIR.mkdir(parents=True, exist_ok=True)
print(f"“Reach”（可达）目录已准备好: {REACH_DIR}")
print(f"“UnReach”（不可达）目录已准备好: {UNREACH_DIR}")

# --- 4. 检查输入目录 ---
if not INPUT_DIR.exists():
    print(f"错误：找不到输入目录 {INPUT_DIR}")
    raise FileNotFoundError("JSON snapshot directory not found")

print(f"\n--- 开始处理 {INPUT_DIR} 中的所有样本 ---")
json_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.json')]

if not json_files:
    print(f"警告：在 {INPUT_DIR} 中未找到任何 .json 文件。")

# --- 5. 遍历、计算、分类并保存 ---

reach_count = 0
unreach_count = 0

for filename in tqdm(json_files, desc="分类健康因子"):
    file_path = INPUT_DIR / filename
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    snapshot_data = data.get('pre_liquidation_snapshot', [])
    
    total_collateral_weighted_eth = Decimal('0')
    total_debt_eth = Decimal('0')
    health_factor_decimal = None
    health_factor_string = "Error"
    target_path = None

    if not snapshot_data:
        health_factor_string = "Error - No Snapshot Data"
        target_path = UNREACH_DIR / filename
        unreach_count += 1
    else:
        # --- 5a. 执行 HF 计算 ---
        try:
            for asset in snapshot_data:
                # 计算债务 (分母)
                debt_amount_raw = Decimal(asset['currentTotalDebt'])
                if debt_amount_raw > 0:
                    decimals = int(asset['reserve']['decimals'])
                    price_in_eth = Decimal(asset['reserve']['price']['priceInEth']) / PRICE_DECIMALS
                    debt_amount = debt_amount_raw / (Decimal('10') ** decimals)
                    total_debt_eth += (debt_amount * price_in_eth)
                
                # 计算抵押品 (分子)
                if asset['usageAsCollateralEnabledOnUser']:
                    collateral_amount_raw = Decimal(asset['currentATokenBalance'])
                    if collateral_amount_raw > 0:
                        decimals = int(asset['reserve']['decimals'])
                        price_in_eth = Decimal(asset['reserve']['price']['priceInEth']) / PRICE_DECIMALS
                        lt_factor = Decimal(asset['reserve']['reserveLiquidationThreshold']) / LT_DECIMALS
                        
                        collateral_amount = collateral_amount_raw / (Decimal('10') ** decimals)
                        collateral_in_eth = collateral_amount * price_in_eth
                        total_collateral_weighted_eth += (collateral_in_eth * lt_factor)
            
            # --- 5b. 计算并分类 ---
            if total_debt_eth == Decimal('0'):
                health_factor_decimal = Decimal('inf')
                health_factor_string = "Infinity"
                target_path = UNREACH_DIR / filename # 无债务样本归为 "UnReach"
                unreach_count += 1
            else:
                health_factor_decimal = total_collateral_weighted_eth / total_debt_eth
                health_factor_string = f"{health_factor_decimal:.18f}" # 保存非科学计数法
                
                # *** 核心分类逻辑 ***
                if health_factor_decimal < Decimal('1.0'):
                    target_path = REACH_DIR / filename # 正常，可清算
                    reach_count += 1
                else:
                    target_path = UNREACH_DIR / filename # 异常，HF >= 1
                    unreach_count += 1
                    
        except Exception as e:
            health_factor_string = f"Error - {e}"
            target_path = UNREACH_DIR / filename # 计算出错归为 "UnReach"
            unreach_count += 1
    
    # --- 5c. 增强 JSON 并保存 ---
    
    # 无论结果如何，都将 HF 添加到数据中
    data['health_factor'] = health_factor_string
    
    # 写入到目标路径
    if target_path:
        with open(target_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
    else:
        print(f"警告：文件 {filename} 未指定目标路径，已跳过。")
        unreach_count += 1 # 理论上不应发生，但作为保险

print("\n--- 分类完成 ---")
print(f"✅ 正常 (Reach, HF < 1) 样本: {reach_count} 个")
print(f"❌ 异常 (UnReach, HF >= 1 或 Error) 样本: {unreach_count} 个")

“Reach”（可达）目录已准备好: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\Reach_and_unReach_sample\Reach
“UnReach”（不可达）目录已准备好: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\Reach_and_unReach_sample\UnReach

--- 开始处理 F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\reserves_picture_PRE_liquidation 中的所有样本 ---


分类健康因子: 100%|██████████| 42/42 [00:00<00:00, 242.30it/s]


--- 分类完成 ---
✅ 正常 (Reach, HF < 1) 样本: 16 个
❌ 异常 (UnReach, HF >= 1 或 Error) 样本: 26 个
